In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np
# Load NIH ChestX-ray14 metadata
# Data_Entry_2017_v2020.csv contains image index
df = pd.read_csv(
    '/content/drive/MyDrive/dissertation/'
    'Data_Entry_2017_v2020.csv')

print(f"Dataset loaded successfully")
print(f"Total rows    : {len(df):,}")
print(f"Total columns : {len(df.columns)}")
print(f"\nColumn names  : {df.columns.tolist()}")

Dataset loaded successfully
Total rows    : 112,120
Total columns : 11

Column names  : ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Sex', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]']


In [10]:
# Basic  about the dataset
print("=" * 50)
print("NIH ChestX-ray14-Dataset Overview")
print("=" * 50)

print(f"\nTotal images    : {len(df):,}")
print(f"Unique patients : {df['Patient ID'].nunique():,}")
print(f"Age range       : {df['Patient Age'].min()}"
      f" — {df['Patient Age'].max()} years")
print(f"Mean age        : {df['Patient Age'].mean():.1f} years")

NIH ChestX-ray14-Dataset Overview

Total images    : 112,120
Unique patients : 30,805
Age range       : 0 — 95 years
Mean age        : 46.6 years


In [11]:
# sex distribution
print("Sex Distribution")
sex_count = df['Patient Sex'].value_counts()
sex_pct   = df['Patient Sex'].value_counts(
                normalize=True) * 100

for s in sex_count.index:
    print(f"  {s} : {sex_count[s]:,} "
          f"({sex_pct[s]:.2f}%)")

Sex Distribution
  M : 63,340 (56.49%)
  F : 48,780 (43.51%)


In [12]:
# age distribution
bins   = [0, 20, 40, 60, 80, 120]
labels = ['0-20', '20-40', '40-60', '60-80', '80+']
df['Age Group'] = pd.cut(
    df['Patient Age'], bins=bins, labels=labels)

print("Age Group Distribution")
age_counts = df['Age Group'].value_counts().sort_index()
for group, count in age_counts.items():
    pct = count / len(df) * 100
    print(f"  {group} : {count:,} ({pct:.1f}%)")

Age Group Distribution
  0-20 : 8,056 (7.2%)
  20-40 : 30,315 (27.0%)
  40-60 : 49,646 (44.3%)
  60-80 : 23,258 (20.7%)
  80+ : 831 (0.7%)


In [13]:
#top conditions / pathology
print("Top 10 Pathology Labels")
all_labels = df['Finding Labels'].str.split('|').explode()
top10 = all_labels.value_counts().head(10)
for label, count in top10.items():
    pct = count / len(df) * 100
    print(f"  {label:<25} : {count:,} ({pct:.1f}%)")

Top 10 Pathology Labels
  No Finding                : 60,361 (53.8%)
  Infiltration              : 19,894 (17.7%)
  Effusion                  : 13,317 (11.9%)
  Atelectasis               : 11,559 (10.3%)
  Nodule                    : 6,331 (5.6%)
  Mass                      : 5,782 (5.2%)
  Pneumothorax              : 5,302 (4.7%)
  Consolidation             : 4,667 (4.2%)
  Pleural_Thickening        : 3,385 (3.0%)
  Cardiomegaly              : 2,776 (2.5%)


In [14]:
#fairness gap by sex
df['No Finding'] = (df['Finding Labels'] == 'No Finding')

print("No Finding Rate by Sex")
print("(Lower rate = more pathology cases)")
nf_sex = df.groupby(
    'Patient Sex', observed=True)['No Finding'].mean() * 100
for s, v in nf_sex.items():
    print(f"  {s} : {v:.2f}%")

gap = abs(nf_sex['M'] - nf_sex['F'])
print(f"\n  Gap between M and F : {gap:.2f}%")

No Finding Rate by Sex
(Lower rate = more pathology cases)
  F : 54.20%
  M : 53.56%

  Gap between M and F : 0.65%


In [15]:
# fairness gap by age
print("No Finding Rate by Age Group")
print("(Lower rate = more pathology cases)")
nf_age = df.groupby(
    'Age Group', observed=True)['No Finding'].mean() * 100
for a, v in nf_age.items():
    print(f"  {a} : {v:.2f}%")

youngest = nf_age['0-20']
oldest   = nf_age['80+']
print(f"\n  Gap between 0-20 and 80+ : "
      f"{abs(youngest - oldest):.2f} percentage points")
print(f"  This motivates fairness-aware FL aggregation")

No Finding Rate by Age Group
(Lower rate = more pathology cases)
  0-20 : 58.97%
  20-40 : 57.72%
  40-60 : 53.24%
  60-80 : 48.67%
  80+ : 42.12%

  Gap between 0-20 and 80+ : 16.86 percentage points
  This motivates fairness-aware FL aggregation


In [16]:
# Partition data across 5 simulated hospital clients
# using Dirichlet label skew Dir(alpha=0.5)

np.random.seed(42)
# Binary label: 1 = pathology present, 0 = No Finding
df['label'] = (df['Finding Labels'] != 'No Finding').astype(int)

NUM_CLIENTS = 5
ALPHA       = 0.5  # Lower = more heterogeneous across clients

# Separate indices by class
idx_0 = np.where(df['label'].values == 0)[0]  # No Finding
idx_1 = np.where(df['label'].values == 1)[0]  # Pathology

np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

# Sample proportions from Dirichlet distribution
# Each client gets a different proportion of each class
props_0 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))
props_1 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))

# Convert proportions to image counts
splits_0 = (props_0 * len(idx_0)).astype(int)
splits_1 = (props_1 * len(idx_1)).astype(int)

# Fix rounding so all images are assigned
splits_0[-1] = len(idx_0) - splits_0[:-1].sum()
splits_1[-1] = len(idx_1) - splits_1[:-1].sum()

# Build client datasets
hospital_names = [
    'Hospital A', 'Hospital B', 'Hospital C',
    'Hospital D', 'Hospital E'
]

clients = {}
ptr0, ptr1 = 0, 0
for i, name in enumerate(hospital_names):
    idx = np.concatenate([
        idx_0[ptr0:ptr0+splits_0[i]],
        idx_1[ptr1:ptr1+splits_1[i]]
    ])
    clients[name] = df.iloc[idx].copy()
    ptr0 += splits_0[i]
    ptr1 += splits_1[i]

print("5 hospital clients created successfully")

5 hospital clients created successfully


In [17]:
# Summary table showing each hospital client
# Confirms non-IID , each client different data

rows = []
for name, data in clients.items():
    rows.append({
        'Client'      : name,
        'Images'      : len(data),
        'No Finding%' : round(
            (data['label']==0).mean()*100, 1),
        'Pathology%'  : round(
            data['label'].mean()*100, 1),
        'Mean Age'    : round(
            data['Patient Age'].mean(), 1),
        'Male%'       : round(
            (data['Patient Sex']=='M').mean()*100, 1),
        'Female%'     : round(
            (data['Patient Sex']=='F').mean()*100, 1),
    })

summary = pd.DataFrame(rows).set_index('Client')

print("NON-IID HOSPITAL PARTITION — Dirichlet Dir(α=0.5)")
print("NIH ChestX-ray14 | 5 Simulated Hospital Clients")
print("Reference: Morafah et al. 2022, Babar et al. 2024")
print("=" * 65)
print(summary.to_string())
print("=" * 65)

rates = [d['label'].mean() for d in clients.values()]
print(f"\nPathology rate range   : "
      f"{min(rates)*100:.1f}% — {max(rates)*100:.1f}%")
print(f"Variance across clients: {np.var(rates):.6f}")
print(f"\nNon-IID confirmed — different distributions")
print(f"across all 5 hospital clients")

NON-IID HOSPITAL PARTITION — Dirichlet Dir(α=0.5)
NIH ChestX-ray14 | 5 Simulated Hospital Clients
Reference: Morafah et al. 2022, Babar et al. 2024
            Images  No Finding%  Pathology%  Mean Age  Male%  Female%
Client                                                               
Hospital A   47664         33.1        66.9      47.1   56.6     43.4
Hospital B    7292          2.2        97.8      47.9   56.3     43.7
Hospital C   10639         15.2        84.8      47.7   57.0     43.0
Hospital D    3201          9.8        90.2      48.1   57.3     42.7
Hospital E   43324         98.0         2.0      45.5   56.3     43.7

Pathology rate range   : 2.0% — 97.8%
Variance across clients: 0.120370

Non-IID confirmed — different distributions
across all 5 hospital clients
